# Nutrition5k: Qwen3-VL-4B (pod)

VLM que cabe en un móvil, como referencia zero-shot (ver `notes/decisions_and_concepts/reference-models.md`). Se
ejecuta con `llama-server` de llama.cpp con los mismos pesos `Q4_0` que Qualcomm AI Hub usa en su runtime llama.cpp
para Snapdragon, y se evalúa con `python -m nutrition5k vlm` (mismo prompt y recorte que los VLMs por API,
`api.ipynb`). La latencia en el móvil no se mide aquí. Cualquier GPU de ≥ 8 GB basta.

Resultados en `notebooks/results_general.ipynb` ("Modelos de referencia").


## Parámetros

In [ ]:
FULL = True  # False: piloto 32/8 para pruebas rápidas

FULL_FLAG = "--full" if FULL else ""
SPLIT = "full" if FULL else "pilot"

## Preparación del entorno

In [ ]:
import os
import sys
from pathlib import Path

# The volume is mounted at /workspace only on RunPod.
RUNPOD = Path("/workspace").is_dir()
if RUNPOD:
    # SSH sessions (VS Code Remote) do not inherit the pod's variables, such as the W&B secret.
    for variable in Path("/proc/1/environ").read_text().split("\0"):
        name, _, value = variable.partition("=")
        if name == "WANDB_API_KEY":
            os.environ.setdefault(name, value)
    # The template points the Hugging Face cache at the volume, where its symlinks break
    # (object storage); keep the pretrained weights on the pod's local disk instead.
    os.environ["HF_HOME"] = "/root/.cache/huggingface"

REPOSITORY_URL = "https://github.com/victorabellotorres/thesis-multimodal-health-ai.git"
# Local disk of the pod: fast, but lost when the pod ends.
REPOSITORY_DIR = Path("/root/thesis-multimodal-health-ai")
BRANCH = "final-experiments"
# Persistent storage: the volume mounted at /workspace.
PERSIST_ROOT = Path("/workspace/nutrition5k")

if RUNPOD:
    if (REPOSITORY_DIR / ".git").is_dir():
        !git -C {REPOSITORY_DIR} fetch
        !git -C {REPOSITORY_DIR} checkout {BRANCH}
        !git -C {REPOSITORY_DIR} pull --ff-only
    else:
        !git clone -b {BRANCH} {REPOSITORY_URL} {REPOSITORY_DIR}
    %cd {REPOSITORY_DIR}
    # The image's system Python is "externally managed" (PEP 668); the pod is disposable.
    %pip install -q --break-system-packages -e .[train]
elif Path.cwd().parent.name == "notebooks":
    # Local: the package is already installed in code/.venv; run from the repository root.
    os.chdir("../..")

# The kernel's interpreter, so `!` commands use the same environment locally (VS Code may not activate .venv).
PYTHON = sys.executable

if RUNPOD:
    # Results live on the volume so nothing is lost when the pod ends.
    (PERSIST_ROOT / "outputs").mkdir(parents=True, exist_ok=True)
    if not Path("outputs").exists():
        Path("outputs").symlink_to(PERSIST_ROOT / "outputs")
print("W&B:", "on" if os.environ.get("WANDB_API_KEY") else "off (only local files)")

In [ ]:
DATA_TAR = PERSIST_ROOT / f"data-{SPLIT}-overhead.tar"

if RUNPOD and DATA_TAR.is_file():
    !tar -xf {DATA_TAR}
else:
    !{PYTHON} -m nutrition5k fetch
    !{PYTHON} -m nutrition5k prepare {FULL_FLAG}
    if RUNPOD and _exit_code == 0:  # a failed prepare must not be cached as a partial .tar
        !tar -cf {DATA_TAR} data
!{PYTHON} -m nutrition5k preprocess  # excluded_dishes.csv and dish_groups.csv
!{PYTHON} -m nutrition5k check {FULL_FLAG}

## Qwen3-VL-4B con llama.cpp

Compila `llama-server` con CUDA (una vez por pod, unos minutos), descarga el modelo `Q4_0` y su proyector de visión
(`mmproj-F16`) de `unsloth/Qwen3-VL-4B-Instruct-GGUF` (Apache-2.0), y pregunta primero 30 platos de validación
(comprobar que la respuesta se puede leer) y después los 507 de test. Muestreo por defecto de `llama-server`.
4 peticiones en paralelo, 4096 tokens de contexto por petición.


In [ ]:
import subprocess
import time
import urllib.request

LLAMA_DIR = Path("/root/llama.cpp")
QWEN_REPO = "unsloth/Qwen3-VL-4B-Instruct-GGUF"
QWEN_NAME = "qwen3-vl-4b-q4_0"  # nombre de la carpeta de resultados en outputs/vlm/

if not (LLAMA_DIR / "build/bin/llama-server").is_file():
    !git clone -q --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_DIR}
    %pip install -q --break-system-packages cmake
    !cmake -S {LLAMA_DIR} -B {LLAMA_DIR}/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF -DCMAKE_CUDA_ARCHITECTURES=native > /root/llama-build.log
    !cmake --build {LLAMA_DIR}/build -j --target llama-server >> /root/llama-build.log
print("llama.cpp", subprocess.run(["git", "-C", str(LLAMA_DIR), "log", "-1", "--format=%h %cs"], capture_output=True, text=True).stdout)

from huggingface_hub import hf_hub_download

model_file = hf_hub_download(QWEN_REPO, "Qwen3-VL-4B-Instruct-Q4_0.gguf")
mmproj_file = hf_hub_download(QWEN_REPO, "mmproj-F16.gguf")
server = subprocess.Popen(
    [LLAMA_DIR / "build/bin/llama-server", "-m", model_file, "--mmproj", mmproj_file, "-ngl", "99",
     "-c", "16384", "-np", "4", "--port", "8080"],
    stdout=open("/root/llama-server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(120):  # loading takes a few seconds
    try:
        if urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=5).status == 200:
            break
    except OSError:
        time.sleep(5)
else:
    raise RuntimeError("llama-server did not start: see /root/llama-server.log")
print("llama-server ready")


In [ ]:
def vlm(model, provider, split, limit=None, workers=4):
    """One VLM run of `python -m nutrition5k vlm`; finished dishes are skipped."""
    command = f"{PYTHON} -m nutrition5k vlm --model {model} --provider {provider} --split {split} --workers {workers} {FULL_FLAG}"
    if limit:
        command += f" --limit {limit}"
    print(command)
    get_ipython().system(command)
    if get_ipython().user_ns["_exit_code"]:
        raise RuntimeError(f"vlm failed: {command}")


vlm(QWEN_NAME, "local", "val", limit=30)
vlm(QWEN_NAME, "local", "test")
server.terminate()
server.wait()
